# 04 · Traducción de Eráviri al español

El notebook `04_Traduccion.ipynb` implementa entrenamiento, inferencia, métricas,
ablaciones y exportación de la entrega. El código está en `lexicon/translation.py`.
No usa modelos externos de traducción ni archivos privados del docente.


Ejecutar en orden. La instalacion se hace en el kernel activo; reiniciarlo si antes cargo binarios de otro entorno. Este notebook puede ejecutarse sin los resultados guardados del 03.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                     if (p / 'ENUNCIADO.md').is_file() and (p / 'datos').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Abre este notebook dentro del proyecto.')
# No reutilizar binarios de .deps pertenecientes a otro Python.
sys.path[:] = [p for p in sys.path if Path(p or '.').resolve() != (PROJECT_ROOT / '.deps').resolve()]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Proyecto:', PROJECT_ROOT)
print('Python del kernel:', sys.version)
print('Interprete:', sys.executable)


Proyecto: C:\Users\aml\Desktop\GestionConocimiento
Python del kernel: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Interprete: C:\Program Files\Python314\python.exe


In [ ]:
requirements_file = str(PROJECT_ROOT / "requirements-traduccion.txt")
%pip install -r "$requirements_file"


## 1. Hipótesis y protocolo

Hipótesis falsable: un traductor que separa vocabulario y estructura logra más
exact matches en combinaciones nuevas que recuperar una traducción completa
por similitud de palabras. La comparación principal usa los mismos 420 pares
de entrenamiento y los mismos 120 pares de validación.

Antes de evaluar se fijan la gramática respaldada por el EDA, el uso de las
formas verbales observadas y la generación española. No se añaden excepciones
por oración de validación. Esta validación ya fue explorada en los cuadernos
anteriores: no es una evaluación ciega independiente. El test oficial oculto
sigue siendo la evaluación pendiente de generalización externa.


In [2]:
from lexicon.preprocessing import prepare
from lexicon.experimental import load_event
from lexicon.translation import (
    CompositionalTranslator, JaccardTranslator, ConversationRouter,
    UnsupportedSentence, export_translation,
)
data, _ = prepare(PROJECT_ROOT)
event = load_event(PROJECT_ROOT)
print('Aprendizaje:', len(data['train']), 'pares; validacion:', len(data['validation']))


Aprendizaje: 420 pares; validacion: 120


## 2. Baseline obligatorio: Jaccard

Se tokeniza la entrada y cada fuente de entrenamiento, se convierten en conjuntos
y se calcula intersección/unión. Se copia la traducción del ejemplo con mayor
similitud. Los empates se resuelven por el primer ejemplo de entrenamiento,
igual que en la plantilla original.

El baseline reconoce elementos comunes, pero no combina traducciones, no usa
el orden y puede cambiar objeto, acción, lugar o negación al recuperar un vecino.


In [3]:
baseline = JaccardTranslator().fit(data['train'])
source = 'telu tavi nemi.'
print('Fuente:', source)
print('Jaccard:', baseline.translate(source))


Fuente: telu tavi nemi.
Jaccard: el maestro recuerda la carta.


## 3. Traductor composicional y qué aprende realmente

Se aplica la hipótesis de estructura de los pares controlados:

```text
Eráviri: SUJETO OBJETO [LUGAR] [nu] VERBO
Español: SUJETO [no] VERBO OBJETO [en LUGAR]
```

Las reglas son explícitas y se justifican con entrenamiento; no se presentan
como una gramática descubierta automáticamente de cualquier idioma. El léxico
sí se induce contando correspondencias de los 420 pares: diez sujetos, doce
objetos, seis lugares y treinta formas verbales. No se almacenan las frases de
validación ni sus traducciones como excepciones.

Las correspondencias con traducciones contradictorias producen un error de
entrenamiento, en vez de elegir silenciosamente una. Todas las fuentes de
entrenamiento deben ser compatibles con la plantilla. Las pistas oficiales
se contrastan después y no se utilizan para ajustar el léxico del experimento.

Ejemplo observado en entrenamiento:

```text
vonu zura miri nu lomuta.
  sujeto: vonu       → el viajero
  objeto: zura       → la llave
  lugar: miri        → en el bosque
  negación: nu       → no
  verbo: lomuta      → buscó

el viajero no buscó la llave en el bosque.
```

Se conservan las conjugaciones españolas observadas: no se generan formas
irregulares concatenando terminaciones. El análisis identifica raíz y tiempo,
pero la realización española consulta la forma verbal completa aprendida. Las
30 formas ya aparecen en entrenamiento. Por tanto, el experimento demuestra
recombinación de formas conocidas, **no** conjugación de un verbo nunca visto.

Una fuente desconocida o fuera de la gramática produce `UnsupportedSentence`.
No se sustituye por una frase parecida que pudiera alterar el sentido. Para
exportar los CSV requerimos cobertura completa de sus cuarenta entradas.


In [4]:
rules = CompositionalTranslator().fit(data['train'])
print('Inventario aprendido:', {k: len(v) for k, v in rules.lexicon.items()})
source = 'vonu zura miri nu lomuta.'
print('Fuente:', source)
print('Analisis:', rules.analyze(source))
print('Traduccion:', rules.translate(source))
print('Datos rechazados de train:', rules.evidence)


Inventario aprendido: {'subject': 10, 'object': 12, 'location': 6, 'verb': 30}
Fuente: vonu zura miri nu lomuta.
Analisis: {'subject': 'vonu', 'object': 'zura', 'location': 'miri', 'negative': True, 'root': 'lomu', 'tense': 'pasado', 'signature': 'S O L NEG V'}
Traduccion: el viajero no buscó la llave en el bosque.
Datos rechazados de train: {'rejected_train_ids': [], 'negation_disagreement_ids': []}


## 4. Métricas y normalización

Exact match usa NFC, minúsculas, espacios uniformes y eliminación de puntuación
final. Se preservan tildes y puntuación interna. Se mide la proporción de frases
completamente iguales a la referencia.

chrF se calcula con **sacrebleu 2.6.0**, sobre las mismas cadenas normalizadas:
orden de caracteres 6, orden de palabras 0, beta 2, sin espacios y con orden
efectivo. Se registra la firma del evaluador y su versión. Su escala es 0–100;
exact match se guarda como proporción 0–1.

No se usa BLEU como indicador principal. chrF permite ver semejanza parcial,
pero puede ser alto aunque falte una negación o cambie el tiempo verbal.


In [5]:
# Fit solo con train; validacion sirve para evaluar. Exporta los CSV al final.
report = export_translation(PROJECT_ROOT)
print('Preprocesamiento:', report['metric_preprocessing'])
print('Configuracion:', report['validation_results']['compositional']['groups']['all']['chrf_signature'])


Preprocesamiento: EM y chrF sobre NFC, minúsculas, espacios normalizados y sin puntuación final; puntuación interna conservada.
Configuracion: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0


## 5. Resultado principal en validación

| Sistema | Aciertos | Exact match | chrF |
|---|---:|---:|---:|
| Jaccard | 0/120 | 0,00 % | 64,63 |
| Composicional completo | 120/120 | 100,00 % | 100,00 |
| Ablación: sin tiempo | 82/120 | 68,33 % | 94,28 |
| Ablación: sin negación | 103/120 | 85,83 % | 98,09 |
| Ablación: sin reordenamiento | 0/120 | 0,00 % | 84,08 |

Los sistemas no se ajustan frase por frase después de observar estos resultados.
La comparación respalda la hipótesis dentro de los datos sintéticos disponibles.
No se atribuye el 100 % a un dominio general del idioma ni se garantiza el test
oculto. La gramática es muy regular y todos los tokens de validación son conocidos.


In [6]:
for system, result in report['validation_results'].items():
    score = result['groups']['all']
    print(f"{system:22} {score['correct']:3}/{score['n']}  EM={100 * score['exact_match']:6.2f}%  chrF={score['chrf']:6.2f}")


jaccard                  0/120  EM=  0.00%  chrF= 64.63
compositional          120/120  EM=100.00%  chrF=100.00
without_tense           82/120  EM= 68.33%  chrF= 94.28
without_negation       103/120  EM= 85.83%  chrF= 98.09
without_reordering       0/120  EM=  0.00%  chrF= 84.08


![01_comparacion_metricas](resultados/traduccion_04/01_comparacion_metricas.png)


## 6. Generalización y análisis por fenómeno

El modelo completo obtiene 17/17 en negación, 16/16 en pasado, 22/22 en futuro,
85/85 con lugar y 35/35 sin lugar. Los grupos se solapan.

Se define terna nueva como sujeto–objeto–raíz verbal ausente de entrenamiento:
el modelo completo obtiene **87/87** en ternas nuevas y **33/33** en ternas vistas.
Jaccard obtiene cero en ambos grupos. La ausencia de la oración completa no se
confunde con vocabulario desconocido ni con un cambio de distribución demostrado.

`metricas_grupos.csv` muestra todos los subconjuntos y sus tamaños. Los grupos
vacíos tienen métricas nulas, no una falsa precisión de cero. No se observa
ninguna abstención del modelo completo en los conjuntos disponibles.


In [7]:
for group, score in report['validation_results']['compositional']['groups'].items():
    print(group, 'n:', score['n'], 'correctas:', score['correct'], 'EM:', score['exact_match'], 'chrF:', score['chrf'])
from lexicon.translation_plots import plot_translation
figures = plot_translation(PROJECT_ROOT)
print('Figuras:', figures)


all n: 120 correctas: 120 EM: 1.0 chrF: 100.0
negation n: 17 correctas: 17 EM: 1.0 chrF: 100.0
past n: 16 correctas: 16 EM: 1.0 chrF: 100.0
future n: 22 correctas: 22 EM: 1.0 chrF: 100.0
present n: 82 correctas: 82 EM: 1.0 chrF: 100.0
location n: 85 correctas: 85 EM: 1.0 chrF: 100.0
without_location n: 35 correctas: 35 EM: 1.0 chrF: 100.0
novel_triple n: 87 correctas: 87 EM: 1.0 chrF: 100.0
seen_triple n: 33 correctas: 33 EM: 1.0 chrF: 100.0
unparsed n: 0 correctas: 0 EM: None chrF: None
Figuras: ['01_comparacion_metricas', '02_fenomenos_composicion', '03_errores_baseline']


![02_fenomenos_composicion](resultados/traduccion_04/02_fenomenos_composicion.png)


## 7. Ablaciones: qué cambia y qué se conserva

Sin tiempo: el analizador sigue identificando la forma de entrada, pero el
generador utiliza la traducción presente de su raíz. Léxico, orden y negación
se conservan. Falla en los 38 casos pasados/futuros y conserva los 82 presentes.
Esto mide la contribución de conservar tiempo, no la capacidad de aprender
conjugaciones nuevas ni una ablación total del analizador morfológico.

Sin negación: solo se omite `no` en la generación. Falla exactamente en las
17 frases negativas. Su chrF de 98,09 ilustra que semejanza superficial alta
no asegura conservación del significado.

Sin reordenamiento: se mantienen las traducciones de cada componente, los
artículos, el tiempo y la negación, pero se colocan según el orden fuente.
La caída de exact match mide el aporte del orden español en esta generación.
Cada ablación cambia un único componente y conserva todo lo demás.


In [8]:
source = 'vonu zura miri nu lomuta.'
print('Completo:', rules.translate(source))
print('Sin tiempo:', rules.translate(source, use_tense=False))
print('Sin negacion:', rules.translate(source, use_negation=False))
print('Sin reordenar:', rules.translate(source, reorder=False))


Completo: el viajero no buscó la llave en el bosque.
Sin tiempo: el viajero no busca la llave en el bosque.
Sin negacion: el viajero buscó la llave en el bosque.
Sin reordenar: el viajero la llave en el bosque no buscó.


## 8. Tres errores reales, sin inventar fallos del modelo completo

El modelo completo no tiene errores en esta validación. Para cumplir el análisis
crítico se estudian errores reales del baseline y de las ablaciones, identificando
siempre el sistema al que pertenecen.

1. **Objeto cambiado — Jaccard**
   Fuente: `telu tavi nemi.`
   Referencia: «el maestro recuerda el alimento.»
   Predicción: «el maestro recuerda la carta.»
   Recupera una oración cercana sin reemplazar el objeto por el de la entrada.

2. **Acción cambiada — Jaccard**
   Fuente: `vonu suni rilo kori.`
   Referencia: «el viajero guarda el libro en el río.»
   Predicción: «el viajero elige el libro en el río.»
   Comparte participantes y lugar, pero comunica una acción diferente.

3. **Negación perdida y objeto cambiado — Jaccard**
   Fuente: `naku kava kuna nu vasi.`
   Referencia: «el niño no comparte la carta en el mercado.»
   Predicción: «el niño comparte el mapa en el mercado.»
   Jaccard no exige conservar ni el objeto ni la polaridad de la entrada.

La taxonomía automática compara campos del español controlado y se usa como
diagnóstico, no como métrica semántica universal. `errores.csv` conserva todos
los casos, incluidas las ablaciones, y `validacion_detalle.csv` conserva los 600
resultados de los cinco sistemas sobre las mismas 120 referencias.


In [9]:
import csv
with (PROJECT_ROOT / 'resultados' / 'traduccion_04' / 'errores.csv').open(encoding='utf-8', newline='') as handle:
    errors = list(csv.DictReader(handle))
baseline_errors = [r for r in errors if r['system'] == 'jaccard']
for row in baseline_errors[:3]:
    print('FUENTE:', row['naviri'])
    print('REF:', row['reference'])
    print('PRED:', row['prediction'])
    print('ERRORES:', row['error_types'], '\n')
print('Errores del sistema completo:', sum(row['system'] == 'compositional' for row in errors))


FUENTE: telu tavi nemi.
REF: el maestro recuerda el alimento.
PRED: el maestro recuerda la carta.
ERRORES: ['objeto'] 

FUENTE: vonu suni rilo kori.
REF: el viajero guarda el libro en el río.
PRED: el viajero elige el libro en el río.
ERRORES: ['accion'] 

FUENTE: naku kava kuna nu vasi.
REF: el niño no comparte la carta en el mercado.
PRED: el niño comparte el mapa en el mercado.
ERRORES: ['objeto', 'negacion'] 

Errores del sistema completo: 0


![03_errores_baseline](resultados/traduccion_04/03_errores_baseline.png)


## 9. Relación con el modelo estadístico del notebook 03

El bigrama de Eráviri modela la probabilidad de una fuente; el traductor actual
modela sus correspondencias y estructura. No se incorpora el LM como un factor
artificial de confianza o un supuesto selector de traducciones: para una misma
fuente, su probabilidad es la misma frente a cualquier candidato español.

El monolingüe ayudó a estudiar estructura y mejoró la estimación probabilística
en el experimento 03. El traductor completo **no** utiliza sus conteos. Por tanto,
no se afirma una mejora de traducción causada por el monolingüe. Una comparación
con y sin un componente que el sistema no usa sería una ablación vacía.


In [10]:
print('Usa monolingue para ajustar reglas:', report['monolingual_used_to_fit_rules'])
print('Usa LM para seleccionar traducciones:', report['lm_used_to_rerank'])
print('Usa pistas para ajustar el lexico:', report['hints_used_to_fit'])


Usa monolingue para ajustar reglas: False
Usa LM para seleccionar traducciones: False
Usa pistas para ajustar el lexico: False


## 10. Expresiones oficiales y exportación

Un enrutador separado reconoce por coincidencia normalizada las 18 expresiones
oficiales y devuelve su traducción publicada. Conserva `?`: `seni?` no se confunde
con `seni`. La ruta `official_phrase_memory` se diferencia de `compositional`.
No se suman esos aciertos a la evaluación de validación: son memoria de ejemplos
conocidos, no una prueba de generalización conversacional. Se conservan las
alternativas oficiales como «Disculpa / Perdón» sin inventar una referencia única.

Se producen dos archivos independientes en la raíz:

- `predicciones.csv`: público original, 40 filas, columnas `id,prediccion`.
- `predicciones_fase5.csv`: fase 5, 40 filas, IDs `F5-...`.

Se comprueban tamaño, IDs únicos, predicciones no vacías, columnas y orden contra
cada archivo fuente. Hay copias y trazas estructurales en `resultados/traduccion_04/`.
No hay referencias para medir exactitud del público ni de fase 5; su cobertura
de análisis no se presenta como precisión verificada. No se mezclan los 80 mensajes.


In [11]:
router = ConversationRouter(rules, event['conversation'])
for source in ['salia neri', 'noku', 'nelo mali', 'pali', 'vonu zura miri nu lomuta.']:
    print(source, '->', router.translate(source))
try:
    router.translate('seni')
except UnsupportedSentence as exc:
    print('Entrada no cubierta seni:', str(exc))

for filename, source_rows in [('predicciones.csv', data['public']), ('predicciones_fase5.csv', event['phase5'])]:
    with (PROJECT_ROOT / filename).open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        predictions = list(reader)
        assert reader.fieldnames == ['id', 'prediccion']
    assert len(predictions) == 40
    assert [r['id'] for r in predictions] == [r['id'] for r in source_rows]
    assert all(r['prediccion'].strip() for r in predictions)
    print(filename, 'verificado:', len(predictions), 'filas')
    for row in predictions[:3]:
        print(row)


salia neri -> {'prediction': 'Buenos días', 'route': 'official_phrase_memory'}
noku -> {'prediction': 'No', 'route': 'official_phrase_memory'}
nelo mali -> {'prediction': 'Un mensaje, por favor', 'route': 'official_phrase_memory'}
pali -> {'prediction': 'Disculpa / Perdón', 'route': 'official_phrase_memory'}
vonu zura miri nu lomuta. -> {'prediction': 'el viajero no buscó la llave en el bosque.', 'route': 'compositional'}
Entrada no cubierta seni: Vocabulario o estructura fuera de la cobertura de entrenamiento
predicciones.csv verificado: 40 filas
{'id': 'NX-0001', 'prediccion': 'el maestro observará el libro en el mercado.'}
{'id': 'NX-0002', 'prediccion': 'la investigadora compartirá el libro.'}
{'id': 'NX-0003', 'prediccion': 'la artista observará la carta en la escuela.'}
predicciones_fase5.csv verificado: 40 filas
{'id': 'F5-0001', 'prediccion': 'la niña no recuerda el agua en el mercado.'}
{'id': 'F5-0002', 'prediccion': 'la investigadora recordó la fruta en la casa.'}
{'id': 'F5

## 11. Reproducibilidad, limitaciones y defensa

`traductor_composicional.json` guarda conteos léxicos y análisis de formas; puede
cargarse sin leer validación. No guarda una tabla de traducciones de validación.
Se registran versiones, configuración de métricas y hashes de los siete CSV.
El algoritmo es determinista; el baseline desempata por orden de entrenamiento.

Las limitaciones son el español y la gramática controlados, vocabulario cerrado,
conjugaciones observadas, una validación ya explorada y ausencia del test oculto.
Con una semana adicional estudiaríamos separación por composiciones más difícil,
formas verbales no observadas, evaluación independiente y alternativas neuronales
o probabilísticas de alineamiento, comparadas con este sistema interpretable.

El resultado cumple la comparación y las ablaciones solicitadas, pero el rigor
de la defensa depende de explicar por qué funciona aquí y dónde deja de estar
respaldado por evidencia. El miniinforme y el registro de herramientas acompañan
al notebook; la evaluación oficial sigue pendiente.


In [12]:
model_path = PROJECT_ROOT / 'resultados' / 'traduccion_04' / 'traductor_composicional.json'
loaded = CompositionalTranslator.load(model_path)
print('Modelo cargado:', loaded.translate('vonu zura miri nu lomuta.'))
print('CSV de entrega:', PROJECT_ROOT / 'predicciones.csv')
print('Fase 5:', PROJECT_ROOT / 'predicciones_fase5.csv')
print('Informe:', PROJECT_ROOT / 'TRADUCCION_04.md')
print('Miniinforme:', PROJECT_ROOT / 'MINIINFORME.md')
print('Registro:', PROJECT_ROOT / 'REGISTRO_HERRAMIENTAS.md')


Modelo cargado: el viajero no buscó la llave en el bosque.
CSV de entrega: C:\Users\aml\Desktop\GestionConocimiento\predicciones.csv
Fase 5: C:\Users\aml\Desktop\GestionConocimiento\predicciones_fase5.csv
Informe: C:\Users\aml\Desktop\GestionConocimiento\TRADUCCION_04.md
Miniinforme: C:\Users\aml\Desktop\GestionConocimiento\MINIINFORME.md
Registro: C:\Users\aml\Desktop\GestionConocimiento\REGISTRO_HERRAMIENTAS.md
